In [2]:
# 主題：使用演算法進行「股價或EPS預測」
# 資料 : 作業區股票價格、EPS（.CSV）
# 不可使用前三次作業之股票
# 備註1 : 可使用python或excel建模
# 備註2 : 可使用其他自行取得的資料

In [3]:
import pandas as pd
import numpy as np
import datetime as dt
from statsmodels.tsa import stattools
from statsmodels.graphics.tsaplots import *

import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline


In [6]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

def evaluate_forecast(y_true, y_pred, model_name="Model"):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((np.array(y_true) - np.array(y_pred)) / np.array(y_true))) * 100
    
    print(f"{model_name}")
    print(f"MAE  : {mae:.4f}")
    print(f"RMSE : {rmse:.4f}")
    print(f"MAPE : {mape:.2f}%")
    
    return {"Model": model_name, "MAE": mae, "RMSE": rmse, "MAPE": mape}

In [7]:
#讀取資料
Price = pd.read_excel("2015-2021 eps.xlsx") 

#顯示數據
Price.head()

,Number,2015-Q1,2015-Q2,2015-Q3,2015-Q4,2016-Q1,2016-Q2,2016-Q3,2016-Q4,2017-Q1,...,2019-Q3,2019-Q4,2020-Q1,2020-Q2,2020-Q3,2020-Q4,2021-Q1,2021-Q2,2021-Q3,2021-Q4
0,1101,0.22,0.61,0.26,0.47,0.06,0.54,0.63,0.49,0.23,...,1.20,1.05,0.55,1.39,1.30,1.08,0.57,1.24,0.59,0.90
1,1110,0.06,0.04,0.10,0.05,0.21,0.03,-0.02,-0.03,0.04,...,0.02,0.00,-0.05,0.02,0.04,0.03,0.03,0.05,0.14,0.03
2,1216,0.79,0.81,0.68,0.20,0.76,0.84,0.73,0.23,0.79,...,0.91,0.49,0.79,1.12,1.26,0.62,0.95,0.95,1.01,0.59
3,1229,0.33,0.84,0.37,0.05,0.38,0.66,0.78,0.55,0.50,...,0.91,0.35,0.20,0.94,0.77,0.52,0.98,1.18,0.62,0.14
4,1231,0.66,0.21,0.80,0.74,0.84,0.51,0.76,0.96,0.47,...,0.60,1.18,0.56,0.77,1.07,1.34,1.02,0.85,1.00,1.16


In [ ]:
#目標： 2330 臺積電
#疫情(2019/10)後預測

#檢查空值
Price.isna().sum().sum()

In [ ]:
#改名
Price1 = Price.rename(columns={"Number":"Date"})

#矩陣轉置
Price2 = Price1.set_index("Date").T
Price2


In [ ]:
#將時間設為變數
Price2.index = pd.to_datetime(Price2.index)
Price2

## 時間序列分析

In [ ]:
Price3 = Price2[[2330]]
Price3.columns =["2330"]  #若沒定義，則下列無法增加 Return行

Price_2330 = Price3.rename(columns={"2330":"2330 臺積電"})

Price_2330


In [ ]:
# 定義驗證集的位置
# 假設我們要拿最後 5 筆資料來驗證模型的準確度
valid = Price_2330.iloc[-5:] 

# 檢查一下 valid 是否正確定義
print("驗證集資料：")
print(valid.head())

In [ ]:
Price_2330['Return'] = Price_2330["2330 臺積電"].pct_change()

Price_2330 = Price_2330.fillna(0)           #因為是要第二個才能有百分比變化，所以把第一個Return的值填入 0 (不然沒有數值)

Price_2330

In [ ]:
Price_2330["Return"].plot(figsize=(15,10))

## 自相關性(acf 和 pacf)

In [ ]:
#自相關係數
acfs = stattools.acf(Price_2330["Return"])
acfs

In [ ]:
#偏自相關係數
pacfs = stattools.pacf(Price_2330["Return"])

pacfs

In [ ]:
#plot() 能夠展示變量的趨勢變化，畫圖
#https://blog.csdn.net/Frank_LJiang/article/details/89363901
#ls為線條風格；lw為限條寬度

plot_acf(Price_2330["Return"],ls='-',lw=2)


In [ ]:
plot_pacf(Price_2330["Return"],ls='-',lw=2)

# 自動Arima

In [ ]:
from pmdarima.arima import auto_arima

#切分訓練集與測試集
train = Price_2330[:20]
valid = Price_2330[19:]        #因為為3個月為一期，中間會有空期，所以要打第19個值後

#取2330股價
training = train["2330 臺積電"]
validation = valid["2330 臺積電"]

In [ ]:
#配置超參數範圍
#使用ADF 讓TEST找到最好的d
#define model

model = auto_arima(training,
                   star_p=1,start_q=1,
                   max_p=2,max_q=2,
                   test = "adf",                #用adf檢定是不是平穩的序列，能找到最好的差分(d)應該要用多少
                   d=None,
                   m=4,
                   seasonal =True,              #可以考慮是不是具有季節性因素，有的話會考慮進去
                   start_p=0,
                   D=None,
                   trace=True,
                   error_action = "ignore",     #因為autoarima有很多的套件組成，是將其他沒用的隱藏掉
                   suppress_warning = True,
                   stepwise=True)

model.fit(training)



In [ ]:
forecast = model.predict(n_periods=9)   
forecast = pd.DataFrame(forecast,index = valid.index, columns=["Prediction"])
forecast

In [ ]:
#平均平方數
rms = np.sqrt(np.mean(np.power((np.array(valid["2330 臺積電"])-np.array(forecast["Prediction"])),2)))

rms

In [ ]:
#畫圖
from matplotlib.pylab import rcParams

rcParams["figure.figsize"] = 20,10

plt.plot(train["2330 臺積電"], label="Train")
plt.plot(valid["2330 臺積電"], label="Valid")
plt.plot(forecast["Prediction"], label="ARIMA Forecast")
plt.title("Auto ARIMA Forecast")
plt.legend()
plt.show()




In [ ]:
# ADF test : 確定Arima模型是否合理，p-value < 0.05：通常代表序列較接近平穩，ARIMA 建模會更合理
from statsmodels.tsa.stattools import adfuller

adf_result = adfuller(Price_2330["Return"])
print("ADF Statistic:", adf_result[0])
print("p-value:", adf_result[1])
print("Critical Values:")
for key, value in adf_result[4].items():
    print(f"   {key}: {value}")

In [ ]:
# Residual Analysis : 看預測誤差是否仍有結構，模型是否還能改進。
residuals = valid["2330 臺積電"] - forecast["Prediction"]

plt.figure(figsize=(15, 5))
plt.plot(residuals)
plt.title("ARIMA Residuals")
plt.show()

plt.figure(figsize=(8, 5))
plt.hist(residuals, bins=20)
plt.title("Residual Distribution")
plt.show()

plot_acf(residuals.dropna(), lags=min(10, len(residuals.dropna())-1))
plt.show()

In [ ]:
# Naive forecast: next value = previous actual value
naive_pred = valid["2330 臺積電"].shift(1)
naive_pred.iloc[0] = train["2330 臺積電"].iloc[-1]

naive_result = evaluate_forecast(valid["2330 臺積電"], naive_pred, "Naive Forecast")

plt.figure(figsize=(15, 6))
plt.plot(train["2330 臺積電"], label="Train")
plt.plot(valid["2330 臺積電"], label="Valid")
plt.plot(naive_pred, label="Naive Forecast")
plt.title("Naive Forecast")
plt.legend()
plt.show()

In [ ]:
# Moving Average Forecast
history = list(train["2330 臺積電"])
ma_pred = []

for actual in valid["2330 臺積電"]:
    pred = np.mean(history[-3:])
    ma_pred.append(pred)
    history.append(actual)

ma_pred = pd.Series(ma_pred, index=valid.index)

ma_result = evaluate_forecast(valid["2330 臺積電"], ma_pred, "Moving Average Forecast")

plt.figure(figsize=(15, 6))
plt.plot(train["2330 臺積電"], label="Train")
plt.plot(valid["2330 臺積電"], label="Valid")
plt.plot(ma_pred, label="Moving Average Forecast")
plt.title("Moving Average Forecast")
plt.legend()
plt.show()

In [ ]:
# Exponential Smoothing
from statsmodels.tsa.holtwinters import ExponentialSmoothing

es_model = ExponentialSmoothing(train["2330 臺積電"], trend=None, seasonal=None)
es_fit = es_model.fit()
es_pred = es_fit.forecast(len(valid))

es_result = evaluate_forecast(valid["2330 臺積電"], es_pred, "Exponential Smoothing")

plt.figure(figsize=(15, 6))
plt.plot(train["2330 臺積電"], label="Train")
plt.plot(valid["2330 臺積電"], label="Valid")
plt.plot(es_pred, label="Exponential Smoothing")
plt.title("Exponential Smoothing Forecast")
plt.legend()
plt.show()

In [ ]:
# 整理比照表
result_df = pd.DataFrame([
    naive_result,
    ma_result,
    es_result
])

# 如果你已經有 ARIMA 的預測結果 forecast["Prediction"]
arima_result = evaluate_forecast(valid["2330 臺積電"], forecast["Prediction"], "Auto ARIMA")
result_df = pd.DataFrame([
    naive_result,
    ma_result,
    es_result,
    arima_result
])

result_df.sort_values("RMSE")